# Fase 2 — Notebook 1: obtención y exploración inicial de los datos

**Proyecto:** Sobreduración en la titulación de la educación superior chilena (2020–2025)
**Equipo:** «Integrante 1», «Integrante 2», «Integrante 3»  ·  **Grupo:** «N»

---

**Objetivo de este notebook (OE2 y OE3).** Consolidar los seis archivos anuales
del SIES en un único conjunto verificado contra las cifras oficiales, describir
su estructura y distribuciones, y **levantar el inventario de problemas de
calidad** que la limpieza deberá resolver en el notebook siguiente.

Este notebook no modifica los datos: solo los carga y los observa. Toda
corrección se realiza en `F2_2_Limpieza_Transformacion.ipynb`, de modo que el
diagnóstico quede separado del tratamiento.

## 1. Entorno de ejecución

In [ ]:
# Celda de arranque: hace importable el paquete `src` sin instalar el proyecto
# y funciona igual si el notebook se abre desde la raiz o desde su subcarpeta.
import sys
from pathlib import Path

RAIZ = Path.cwd()
if not (RAIZ / "src").exists():
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ))

import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 140)

from src import config

print("Raiz del proyecto:", RAIZ)
for clave, valor in config.describir_entorno().items():
    print(f"  {clave:12s} {valor}")

## 2. Obtención y consolidación

Los seis CSV suman ~955 MB, un volumen que hace inviable la lectura ingenua con
`pd.read_csv` sobre el archivo completo. La estrategia implementada en
`src/ingesta.py` es:

1. **Lectura por bloques** de 200.000 filas (`chunksize`), de modo que la memoria
   ocupada no depende del tamaño del archivo.
2. **Lectura como texto** y conversión posterior controlada: si un valor no es
   convertible, se registra como coerción en lugar de abortar la carga completa.
3. **Selección de columnas** (`usecols`): 30 de las 40 originales, según el
   esquema declarado en `src/config.py`.
4. **Persistencia en Parquet**, formato columnar y comprimido que conserva los
   tipos y evita volver a parsear texto en cada ejecución.

In [ ]:
from src import ingesta

import time

inicio = time.perf_counter()

if config.PARQUET_CONSOLIDADO.exists():
    df = ingesta.cargar_consolidado()
    bitacora_ingesta = None
    print(f"Parquet consolidado existente cargado en {time.perf_counter()-inicio:.1f} s")
else:
    df, bitacora_ingesta = ingesta.consolidar()

print(f"Registros consolidados: {len(df):,}")
print(f"Columnas: {df.shape[1]}")
print(f"Memoria en RAM: {df.memory_usage(deep=True).sum()/1024**2:,.1f} MB")

### 2.1 Verificación de integridad contra la fuente oficial

El documento de esquema del SIES publica el número exacto de observaciones por
año. Comparar ambos conteos es la prueba de que **no se perdió ni se duplicó
ningún registro** durante la lectura por bloques.

In [ ]:
conteo = df["cat_periodo"].value_counts().sort_index()

verificacion = pd.DataFrame({
    "anio": conteo.index.astype(int),
    "filas_cargadas": conteo.to_numpy(),
    "filas_oficiales_sies": [config.FILAS_OFICIALES_SIES[int(a)] for a in conteo.index],
})
verificacion["diferencia"] = (
    verificacion["filas_cargadas"] - verificacion["filas_oficiales_sies"]
)
verificacion["estado"] = verificacion["diferencia"].map(
    lambda d: "OK" if d == 0 else "REVISAR"
)

print(verificacion.to_string(index=False))
assert (verificacion["diferencia"] == 0).all(), "La carga no coincide con la fuente oficial"
print(f"\nIntegridad verificada: {verificacion['filas_cargadas'].sum():,} registros, "
      "coincidencia exacta con las cifras publicadas por el SIES.")

## 3. Estructura del conjunto de datos

In [ ]:
estructura = pd.DataFrame({
    "columna": df.columns,
    "tipo": [str(t) for t in df.dtypes],
    "no_nulos": df.notna().sum().to_numpy(),
    "valores_unicos": [df[c].nunique(dropna=True) for c in df.columns],
    "memoria_mb": (df.memory_usage(deep=True)[1:] / 1024**2).round(2).to_numpy(),
})
print(estructura.to_string(index=False))

### 3.1 Primeras filas

In [ ]:
df.head(5)

### 3.2 Diccionario de variables

Definiciones tomadas del esquema de registro oficial del SIES (secciones 1.2 y
Anexo I). Se documentan aquí para que el notebook sea autocontenido.

In [ ]:
diccionario = pd.DataFrame(
    [
        ("cat_periodo", "Año académico del proceso de titulación", "2020-2025"),
        ("mrun", "Identificador anonimizado del estudiante", "entero"),
        ("gen_alu", "Sexo del estudiante", "1: Hombre; 2: Mujer"),
        ("fec_nac_alu", "Fecha de nacimiento (AAAAMM)", "190001 = sin información"),
        ("anio_ing_carr_ori", "Año de ingreso a la carrera de origen",
         "9995/9998/9999/1900 = códigos especiales"),
        ("sem_ing_carr_ori", "Semestre de ingreso a la carrera de origen", "1 o 2"),
        ("fecha_obtencion_titulo", "Fecha de obtención del título (AAAAMMDD)",
         "19000101 = sin información"),
        ("tipo_inst_1", "Tipo de institución", "Universidad / IP / CFT"),
        ("nivel_global", "Nivel del programa", "Pregrado / Postgrado / Postítulo"),
        ("dur_estudio_carr", "Duración TEÓRICA de estudios del plan (semestres)", "entero"),
        ("dur_proceso_tit", "Duración TEÓRICA del proceso de titulación (semestres)", "entero"),
        ("dur_total_carr", "Duración TEÓRICA total del plan (semestres)", "entero"),
        ("region_sede", "Región donde se imparte la carrera", "16 regiones"),
        ("jornada", "Jornada del programa", "Diurna/Vespertina/A distancia/..."),
        ("modalidad", "Modalidad de impartición", "Presencial/Semipresencial/No presencial"),
        ("area_conocimiento", "Área MINEDUC (adaptación CINE-UNESCO 1997)", "10 áreas"),
        ("cine_f_13_area", "Área CINE-F 2013 (UNESCO/OCDE)", "clasificación internacional"),
    ],
    columns=["variable", "descripción", "valores"],
)
print(diccionario.to_string(index=False))

> **Advertencia metodológica central.** Las tres variables `dur_*` son
> **duraciones teóricas del plan de estudios**, no el tiempo que tardó el
> estudiante. El dato de trayectoria real **no existe en la base**: debe
> reconstruirse a partir del año y semestre de ingreso y de la fecha del
> título. Esa reconstrucción es la contribución técnica de la Fase 2 y se
> desarrolla en el notebook siguiente.

## 4. Estadísticos descriptivos

In [ ]:
numericas = ["dur_estudio_carr", "dur_proceso_tit", "dur_total_carr",
             "anio_ing_carr_ori", "sem_ing_carr_ori", "gen_alu"]
df[numericas].describe().round(2)

### 4.1 Distribución de las variables categóricas principales

In [ ]:
for columna in ["nivel_global", "tipo_inst_1", "modalidad", "jornada"]:
    conteo = df[columna].value_counts()
    porcentaje = (100 * conteo / len(df)).round(1)
    resumen = pd.DataFrame({"registros": conteo, "porcentaje": porcentaje})
    print(f"--- {columna} ---")
    print(resumen.to_string())
    print()

## 5. Diagnóstico de calidad: valores faltantes y códigos centinela

El conjunto **no presenta celdas vacías en la mayoría de las columnas**, pero
eso no significa que no falte información: el SIES codifica la ausencia con
valores especiales (9999, 19000101, semestre 0). Tratarlos como números reales
produciría duraciones de miles de semestres, por lo que primero hay que
cuantificarlos.

In [ ]:
from src import limpieza

print("Valores faltantes explícitos (celdas vacías en el archivo original):")
faltantes = limpieza.reporte_faltantes(df)
print(faltantes.loc[faltantes["nulos"] > 0].to_string(index=False))
if (faltantes["nulos"] == 0).all():
    print("  (ninguno)")

In [ ]:
centinelas = pd.DataFrame([
    {
        "columna": "anio_ing_carr_ori",
        "codigo": "1900 / 9995 / 9998 / 9999",
        "significado": "Sin información u origen en otro programa o institución",
        "registros": int(df["anio_ing_carr_ori"].isin(config.SENTINELAS_ANIO_INGRESO).sum()),
    },
    {
        "columna": "sem_ing_carr_ori",
        "codigo": "0",
        "significado": "Semestre no informado",
        "registros": int((~df["sem_ing_carr_ori"].isin(config.SEMESTRES_VALIDOS)).sum()),
    },
    {
        "columna": "fecha_obtencion_titulo",
        "codigo": "19000101",
        "significado": "Fecha por defecto (sin información)",
        "registros": int((df["fecha_obtencion_titulo"] <= config.SENTINELA_FECHA).sum()),
    },
    {
        "columna": "fec_nac_alu",
        "codigo": "190001",
        "significado": "Fecha de nacimiento por defecto",
        "registros": int((df["fec_nac_alu"] <= config.SENTINELA_FEC_NAC).sum()),
    },
    {
        "columna": "mrun",
        "codigo": "(vacío)",
        "significado": "Estudiante sin identificador; no documentado por el SIES",
        "registros": int(df["mrun"].isna().sum()),
    },
    {
        "columna": "cod_carrera",
        "codigo": "(vacío)",
        "significado": "Programa sin código; no documentado por el SIES",
        "registros": int(df["cod_carrera"].isna().sum()),
    },
])
centinelas["porcentaje"] = (100 * centinelas["registros"] / len(df)).round(3)
print(centinelas.to_string(index=False))

### 5.1 Duplicados potenciales

Un mismo estudiante puede aparecer más de una vez de forma legítima (dos
títulos distintos). Lo que sí constituye un duplicado es la repetición exacta de
la combinación **estudiante + carrera + fecha de titulación**.

In [ ]:
clave = ["cat_periodo", "mrun", "cod_carrera", "fecha_obtencion_titulo"]
identificables = df.dropna(subset=["mrun"])

duplicados_clave = int(identificables.duplicated(subset=clave).sum())
estudiantes_multi = int(
    (identificables.groupby(["cat_periodo", "mrun"], observed=True).size() > 1).sum()
)

print(f"Duplicados sobre {' + '.join(clave)}: {duplicados_clave:,}")
print(f"Estudiantes con más de un título en un mismo año: {estudiantes_multi:,}")
print(f"Filas sin identificador (excluidas del cotejo): {int(df['mrun'].isna().sum()):,}")

### 5.2 Coherencia interna de las duraciones teóricas

Cabe esperar que `dur_estudio_carr + dur_proceso_tit = dur_total_carr`. La
comprobación revela que **no siempre se cumple**, un hallazgo relevante para
decidir cuál de las tres variables usar como referencia.

In [ ]:
suma = df["dur_estudio_carr"] + df["dur_proceso_tit"]
incoherentes = df.loc[suma != df["dur_total_carr"]]

print(f"Registros donde la suma no coincide: {len(incoherentes):,} "
      f"({100*len(incoherentes)/len(df):.1f}%)")
print("\nPatrones más frecuentes (estudio + proceso = total):")
patron = (
    incoherentes.assign(
        patron=incoherentes["dur_estudio_carr"].astype(str) + " + "
        + incoherentes["dur_proceso_tit"].astype(str) + " = "
        + incoherentes["dur_total_carr"].astype(str)
    )["patron"].value_counts().head(6)
)
print(patron.to_string())
print("\nInterpretación: en estos casos el proceso de titulación ya está contenido")
print("dentro de la duración de estudios declarada, por lo que dur_total_carr es la")
print("única referencia consistente para medir la duración teórica del plan.")

## 6. Exploración visual

Las figuras se generan con `src/viz.py` para mantener un estilo único y quedan
guardadas en `reports/figures/`, desde donde el informe las referencia.

In [ ]:
from src import viz

viz.aplicar_estilo()

titulados_anio = df["cat_periodo"].value_counts().sort_index()
fig = viz.barras(
    titulados_anio,
    "Titulados de educación superior por año del proceso (2020-2025)",
    "Titulados",
    horizontal=False,
    formato="{:,.0f}",
    nombre_archivo="f2_1_titulados_por_anio",
)
print(titulados_anio.to_string())

El descenso de 2020 (203.598 registros, un 27,7 % menos que 2021) refleja la
interrupción de la actividad presencial durante la pandemia, que retrasó
exámenes de grado y ceremonias de titulación hacia el año siguiente.

In [ ]:
fig = viz.barras(
    df["area_conocimiento"].value_counts(),
    "Titulados por área de conocimiento (2020-2025)",
    "Titulados",
    formato="{:,.0f}",
    nombre_archivo="f2_1_titulados_por_area",
)

In [ ]:
fig = viz.barras(
    df["region_sede"].value_counts(),
    "Titulados por región de la sede (2020-2025)",
    "Titulados",
    color=config.PALETA["acento"],
    formato="{:,.0f}",
    nombre_archivo="f2_1_titulados_por_region",
)
concentracion = 100 * df["region_sede"].value_counts(normalize=True).head(3).sum()
print(f"Las tres regiones con más titulados concentran el {concentracion:.1f}% del total.")

In [ ]:
distribucion_dur = df.loc[df["nivel_global"] == "Pregrado", "dur_total_carr"]
fig = viz.histograma(
    distribucion_dur,
    "Duración teórica total de los planes de pregrado",
    "Semestres del plan de estudios",
    bins=24,
    referencia=None,
    nombre_archivo="f2_1_duracion_teorica",
)
print(distribucion_dur.describe().round(2).to_string())

## 7. Inventario de problemas de calidad detectados

Este es el producto principal del notebook: la lista de decisiones que la etapa
de limpieza deberá tomar, con su justificación y su tratamiento previsto.

In [ ]:
hallazgos = pd.DataFrame([
    ("H1", "Códigos centinela en año y semestre de ingreso",
     f"{int(df['anio_ing_carr_ori'].isin(config.SENTINELAS_ANIO_INGRESO).sum()):,} registros",
     "Convertir a NA y descartar: sin fecha de inicio no hay duración que medir"),
    ("H2", "Identificador de estudiante vacío",
     f"{int(df['mrun'].isna().sum()):,} registros",
     "Conservar, pero excluir del cotejo de duplicados"),
    ("H3", "Código de carrera vacío",
     f"{int(df['cod_carrera'].isna().sum()):,} registros",
     "Conservar: el análisis usa el nombre y el área, no el código"),
    ("H4", "Duraciones teóricas internamente incoherentes",
     f"{int((df['dur_estudio_carr'] + df['dur_proceso_tit'] != df['dur_total_carr']).sum()):,} registros",
     "Usar dur_total_carr como única referencia teórica"),
    ("H5", "Posgrado y postítulo mezclados con pregrado",
     f"{int((df['nivel_global'] != 'Pregrado').sum()):,} registros",
     "Filtrar: planes no comparables en duración"),
    ("H6", "Categorías de texto con formato heterogéneo",
     "18 columnas categóricas",
     "Normalizar espacios y capitalización antes de agrupar"),
    ("H7", "Ausencia de la variable de duración real",
     "no existe en la fuente",
     "Derivarla desde ingreso y fecha de titulación"),
], columns=["id", "hallazgo", "magnitud", "tratamiento previsto"])

print(hallazgos.to_string(index=False))
hallazgos.to_csv(config.TABLES_DIR / "f2_1_hallazgos_calidad.csv", index=False)
print(f"\nTabla exportada a {config.TABLES_DIR / 'f2_1_hallazgos_calidad.csv'}")

## 8. Cierre del notebook

**Resultado.** Se consolidaron 1.710.167 registros de seis años con coincidencia
exacta contra las cifras oficiales del SIES, se documentó el esquema de 30
variables y se identificaron siete problemas de calidad con su tratamiento
previsto.

**Hallazgo metodológico más relevante.** La base no contiene la duración real de
las trayectorias: solo la duración teórica de los planes. La variable central
del proyecto debe construirse.

> **Continuar en:** `F2/F2_2_Limpieza_Transformacion.ipynb`